In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run ''util_path''

In [0]:
dbutils.widgets.text("catalog", "ecommerce", "Catalog")
dbutils.widgets.text("data_source", "sellers", "Data Source")

In [0]:
catalog= dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path=f'dataset path/{data_source}'
print(base_path)

In [0]:
df_bronze= spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
display(df_bronze.limit(10))

In [0]:
df_bronze=df_bronze.withColumn(
    'seller_city',
    F.initcap(df_bronze.seller_city)
)

In [0]:
df_bronze.groupBy('seller_id').count().filter('count>1').display()

In [0]:
df_bronze.display()

In [0]:
df_bronze.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
df_silver=  spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")

df_gold= df_silver.select('seller_id', 'seller_city', 'seller_state')


In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")